# Method comparison: outline extraction, clustering, and splitting

Side-by-side comparison of every grouping/splitting method introduced in this package, arranged in the same overall pipeline shape as `visual-pipeline/examples/model_runner.ipynb` (mask-vertices -> group/outline -> split -> visualize) -- but driven by reproducible synthetic data instead of a real model and image, so it runs standalone.

**The scenario**: several "bunches", each a tight cluster of small circular "berries" -- mirroring the real use case this package was built for (grape-bunch mask contours from an ML detector, where each berry is already its own individuated detection). Every method below is compared on the exact same underlying data.

1. **Outline extraction**: `ChiOutline` vs `AlphaOutline` -- the latter keeps interior holes.
2. **Clustering (grouping)**: pooling every point into one cloud and grouping via `ChiOutline`'s own edge-threshold, vs `ObjectClusterGraph`'s object-aware DTSCAN adaptation.
3. **Splitting (separation)**: `BottleneckSplit` (polygon-boundary-concavity-based) vs `GraphBottleneckSplit` (object-graph-based).
4. **Full pipeline, old vs new**: the exact failure mode that motivated all of this -- `ChiOutline` + `BottleneckSplit` over-segmenting a single bunch's own berry-shape noise into dozens of spurious pieces, fixed by `ObjectClusterGraph` + `GraphBottleneckSplit`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, Checkbox

from concavehull import (
    AlphaOutline,
    BottleneckCostCriterion,
    BottleneckSplit,
    ChiOutline,
    GraphBottleneckCostCriterion,
    GraphBottleneckSplit,
    ObjectClusterGraph,
)

## Shared synthetic data generator

Each bunch is a tight cluster of `berries_per_bunch` small circular contours. `ring_mode` arranges one bunch's berries in a ring around an empty center instead of a solid blob -- a genuine hole, which `ChiOutline` cannot represent (see its own `count_boundary_cycles()` docstring) but `AlphaOutline` can.

In [2]:
def make_bunch(rng, center, n_berries, spread, berry_radius, n_contour_pts=8, ring=False):
    """One bunch: n_berries small circular contours clustered around center
    (or arranged in a ring around it, if ring=True)."""
    objects = []
    for i in range(n_berries):
        if ring:
            theta_c = 2 * np.pi * i / n_berries
            bcx = center[0] + spread * np.cos(theta_c) + rng.normal(0, spread * 0.05)
            bcy = center[1] + spread * np.sin(theta_c) + rng.normal(0, spread * 0.05)
        else:
            bcx = center[0] + rng.normal(0, spread)
            bcy = center[1] + rng.normal(0, spread)
        theta = np.linspace(0, 2 * np.pi, n_contour_pts, endpoint=False)
        objects.append(np.stack([bcx + berry_radius * np.cos(theta), bcy + berry_radius * np.sin(theta)], axis=1))
    return objects


def make_scene(n_bunches, berries_per_bunch, bunch_separation, berry_spread, berry_radius, ring_mode, seed):
    rng = np.random.default_rng(seed)
    centers = [(i * bunch_separation, (i % 2) * bunch_separation * 0.4) for i in range(n_bunches)]
    object_points = []
    bunch_id = []
    for bi, c in enumerate(centers):
        ring = ring_mode and bi == 0  # only the first bunch demonstrates a hole
        objs = make_bunch(rng, c, berries_per_bunch, berry_spread, berry_radius, ring=ring)
        object_points.extend(objs)
        bunch_id.extend([bi] * len(objs))
    return object_points, np.array(bunch_id)


def plot_polygon(ax, poly, color, linewidth=2):
    closed = np.vstack([poly, poly[:1]])
    ax.plot(closed[:, 0], closed[:, 1], color=color, linewidth=linewidth)


def plot_groups(ax, object_points, groups, title):
    colors = plt.cm.tab10.colors
    for i, group in enumerate(groups):
        color = colors[i % len(colors)]
        for obj_idx in group:
            plot_polygon(ax, object_points[obj_idx], color)
    all_pts = np.vstack(object_points)
    ax.set_xlim(all_pts[:, 0].min() - 2, all_pts[:, 0].max() + 2)
    ax.set_ylim(all_pts[:, 1].min() - 2, all_pts[:, 1].max() + 2)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)

## 1. Outline extraction: `ChiOutline` vs `AlphaOutline`

Enable `ring_mode` to see the difference: `ChiOutline` forces a single simple polygon (no holes are possible by construction), while `AlphaOutline` reports the ring's interior gap as an actual hole.

In [3]:
@interact(
    n_berries=IntSlider(value=18, min=6, max=30, description="n_berries", continuous_update=False),
    berry_spread=FloatSlider(value=4.0, min=1.0, max=8.0, step=0.5, description="spread", continuous_update=False),
    berry_radius=FloatSlider(value=0.5, min=0.2, max=1.0, step=0.1, description="berry_radius",
                              continuous_update=False),
    chi=FloatSlider(value=1.0, min=-1.0, max=3.0, step=0.1, description="chi", continuous_update=False),
    ring_mode=Checkbox(value=True, description="ring_mode"),
    seed=IntSlider(value=0, min=0, max=20, description="seed", continuous_update=False),
)
def compare_outline(n_berries, berry_spread, berry_radius, chi, ring_mode, seed):
    rng = np.random.default_rng(seed)
    objects = make_bunch(rng, (0, 0), n_berries, berry_spread, berry_radius, ring=ring_mode)
    pts = np.vstack(objects)

    chi_outline = ChiOutline(pts, chi=chi)
    chi_polys = chi_outline.compute()

    alpha_outline = AlphaOutline(pts, chi=chi)
    alpha_polys = alpha_outline.compute()

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].scatter(pts[:, 0], pts[:, 1], s=4, color="gray", alpha=0.4, zorder=1)
    for p in chi_polys:
        plot_polygon(axes[0], p, "tab:blue")
    axes[0].set_aspect("equal")
    axes[0].set_xticks([])
    axes[0].set_yticks([])
    axes[0].set_title(f"ChiOutline -> {len(chi_polys)} polygon(s), 0 hole(s) (can't represent any)")

    axes[1].scatter(pts[:, 0], pts[:, 1], s=4, color="gray", alpha=0.4, zorder=1)
    n_holes = 0
    for p in alpha_polys:
        plot_polygon(axes[1], p["exterior"], "tab:orange")
        for h in p["holes"]:
            plot_polygon(axes[1], h, "tab:red")
            n_holes += 1
    axes[1].set_aspect("equal")
    axes[1].set_xticks([])
    axes[1].set_yticks([])
    axes[1].set_title(f"AlphaOutline -> {len(alpha_polys)} polygon(s), {n_holes} hole(s)")

    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=18, continuous_update=False, description='n_berries', max=30, min=6), Fl…

## 2. Clustering: pooled points (`ChiOutline`) vs object-aware (`ObjectClusterGraph`)

Several well-separated bunches, each a tight cluster of berries. The pooled-point approach dumps every berry's contour into one cloud before grouping -- two bunches that happen to share even one close point pair can get chained together (`ChiOutline`'s edge-removal is structurally a point-level single-linkage threshold). `ObjectClusterGraph` decides membership per *object*, so a single berry contributes no internal shape to confuse the decision, and its z-score-normalized edge thresholding (DTSCAN, Kim & Cho 2019) specifically resists chaining.

In [4]:
@interact(
    n_bunches=IntSlider(value=4, min=2, max=6, description="n_bunches", continuous_update=False),
    berries_per_bunch=IntSlider(value=10, min=5, max=20, description="berries/bunch", continuous_update=False),
    bunch_separation=FloatSlider(value=14.0, min=6.0, max=24.0, step=1.0, description="bunch_sep",
                                  continuous_update=False),
    berry_spread=FloatSlider(value=1.3, min=0.5, max=3.0, step=0.1, description="berry_spread",
                              continuous_update=False),
    chi=FloatSlider(value=0.5, min=-1.0, max=2.0, step=0.1, description="chi (ChiOutline)",
                     continuous_update=False),
    area_z_threshold=FloatSlider(value=1.0, min=-2.0, max=3.0, step=0.1, description="area_z_thr (DTSCAN)",
                                  continuous_update=False),
    length_z_threshold=FloatSlider(value=1.0, min=-2.0, max=3.0, step=0.1, description="length_z_thr (DTSCAN)",
                                    continuous_update=False),
    min_pts=IntSlider(value=3, min=1, max=8, description="min_pts (DTSCAN)", continuous_update=False),
    seed=IntSlider(value=0, min=0, max=20, description="seed", continuous_update=False),
)
def compare_clustering(n_bunches, berries_per_bunch, bunch_separation, berry_spread, chi, area_z_threshold,
                        length_z_threshold, min_pts, seed):
    object_points, true_bunch_id = make_scene(n_bunches, berries_per_bunch, bunch_separation, berry_spread, 0.5,
                                               False, seed)
    pooled = np.vstack(object_points)

    chi_outline = ChiOutline(pooled, chi=chi)
    chi_regions = chi_outline.compute()

    ocg = ObjectClusterGraph(object_points, area_z_threshold, length_z_threshold, min_pts)
    object_clusters = ocg.clusters()

    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    colors = plt.cm.tab10.colors
    axes[0].scatter(pooled[:, 0], pooled[:, 1], s=3, color="gray", alpha=0.3, zorder=1)
    for i, region in enumerate(chi_regions):
        plot_polygon(axes[0], region, colors[i % len(colors)])
    axes[0].set_aspect("equal")
    axes[0].set_xticks([])
    axes[0].set_yticks([])
    axes[0].set_title(f"Pooled points + ChiOutline -> {len(chi_regions)} region(s)\n(true bunch count: {n_bunches})")

    plot_groups(axes[1], object_points, object_clusters,
                f"ObjectClusterGraph (DTSCAN) -> {len(object_clusters)} cluster(s)\n(true bunch count: {n_bunches})")

    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=4, continuous_update=False, description='n_bunches', max=6, min=2), IntS…

## 3. Splitting: `BottleneckSplit` vs `GraphBottleneckSplit`

One already-merged clump of `berries_per_bunch` berries (a single bunch, so the *correct* answer is "don't split further"). `BottleneckSplit` operates on the clump's boundary contour, where individual berries' own round bumps can be misread as a neck between separate objects. `GraphBottleneckSplit` operates on the object graph directly, where a single berry is just one node with no internal shape to misread.

In [5]:
@interact(
    berries_per_bunch=IntSlider(value=10, min=5, max=20, description="n_berries", continuous_update=False),
    berry_spread=FloatSlider(value=1.3, min=0.5, max=3.0, step=0.1, description="berry_spread",
                              continuous_update=False),
    chi=FloatSlider(value=0.8, min=-1.0, max=2.0, step=0.1, description="chi (outline)", continuous_update=False),
    bottleneck_cost_threshold=FloatSlider(value=0.45, min=0.05, max=0.95, step=0.05,
                                           description="cost_thr (Bottleneck)", continuous_update=False),
    graph_cost_threshold=FloatSlider(value=0.1, min=0.02, max=0.5, step=0.02,
                                      description="cost_thr (GraphBottleneck)", continuous_update=False),
    seed=IntSlider(value=0, min=0, max=20, description="seed", continuous_update=False),
)
def compare_splitting(berries_per_bunch, berry_spread, chi, bottleneck_cost_threshold, graph_cost_threshold, seed):
    rng = np.random.default_rng(seed)
    object_points = make_bunch(rng, (0, 0), berries_per_bunch, berry_spread, 0.5)
    pooled = np.vstack(object_points)

    chi_outline = ChiOutline(pooled, chi=chi)
    regions = chi_outline.compute()
    bs = BottleneckSplit(BottleneckCostCriterion(bottleneck_cost_threshold))
    bottleneck_pieces = bs.split_all(regions) if regions else []

    ocg = ObjectClusterGraph(object_points, 1.0, 1.0, 3)
    clusters = ocg.clusters()
    gbs = GraphBottleneckSplit(GraphBottleneckCostCriterion(graph_cost_threshold, 0.1))
    graph_groups = []
    for c in clusters:
        graph_groups.extend(gbs.split(c, ocg) if len(c) >= 2 else [c])

    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    colors = plt.cm.tab10.colors
    axes[0].scatter(pooled[:, 0], pooled[:, 1], s=3, color="gray", alpha=0.3, zorder=1)
    for i, piece in enumerate(bottleneck_pieces):
        plot_polygon(axes[0], piece, colors[i % len(colors)])
    axes[0].set_aspect("equal")
    axes[0].set_xticks([])
    axes[0].set_yticks([])
    axes[0].set_title(f"BottleneckSplit -> {len(bottleneck_pieces)} piece(s)\n(this is ONE bunch -- ideal answer: 1)")

    plot_groups(axes[1], object_points, graph_groups,
                f"GraphBottleneckSplit -> {len(graph_groups)} group(s)\n(this is ONE bunch -- ideal answer: 1)")

    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=10, continuous_update=False, description='n_berries', max=20, min=5), Fl…

## 4. Full pipeline, old vs new

Arranged exactly like `model_runner.ipynb`'s own pipeline shape: mask vertices -> group -> split -> visualize.

- **Old** (steps 6-7 there): pool every berry's contour -> `ChiOutline` -> `BottleneckSplit`.
- **New** (step 8 there): per-object points -> `ObjectClusterGraph` -> `GraphBottleneckSplit` -> (optionally) `AlphaOutline` per final group, for a hole-capable rendered outline.

This is the scenario that motivated this whole comparison: on real mask data, the old pipeline produced 86 spurious pieces for what was visually under 10 bunches.

In [6]:
@interact(
    n_bunches=IntSlider(value=4, min=2, max=6, description="n_bunches", continuous_update=False),
    berries_per_bunch=IntSlider(value=10, min=5, max=20, description="berries/bunch", continuous_update=False),
    bunch_separation=FloatSlider(value=14.0, min=6.0, max=24.0, step=1.0, description="bunch_sep",
                                  continuous_update=False),
    berry_spread=FloatSlider(value=1.3, min=0.5, max=3.0, step=0.1, description="berry_spread",
                              continuous_update=False),
    ring_mode=Checkbox(value=False, description="ring_mode (bunch 0)"),
    chi=FloatSlider(value=0.8, min=-1.0, max=2.0, step=0.1, description="chi (old pipeline)",
                     continuous_update=False),
    bottleneck_cost_threshold=FloatSlider(value=0.45, min=0.05, max=0.95, step=0.05,
                                           description="cost_thr (old)", continuous_update=False),
    graph_cost_threshold=FloatSlider(value=0.1, min=0.02, max=0.5, step=0.02,
                                      description="cost_thr (new)", continuous_update=False),
    render_alpha_outline=Checkbox(value=True, description="render AlphaOutline per group (new pipeline)"),
    seed=IntSlider(value=0, min=0, max=20, description="seed", continuous_update=False),
)
def compare_full_pipeline(n_bunches, berries_per_bunch, bunch_separation, berry_spread, ring_mode, chi,
                           bottleneck_cost_threshold, graph_cost_threshold, render_alpha_outline, seed):
    object_points, true_bunch_id = make_scene(n_bunches, berries_per_bunch, bunch_separation, berry_spread, 0.5,
                                               ring_mode, seed)
    pooled = np.vstack(object_points)

    # old pipeline
    chi_outline = ChiOutline(pooled, chi=chi)
    regions = chi_outline.compute()
    bs = BottleneckSplit(BottleneckCostCriterion(bottleneck_cost_threshold))
    old_pieces = bs.split_all(regions) if regions else []

    # new pipeline
    ocg = ObjectClusterGraph(object_points, 1.0, 1.0, 3)
    clusters = ocg.clusters()
    gbs = GraphBottleneckSplit(GraphBottleneckCostCriterion(graph_cost_threshold, 0.1))
    new_groups = []
    for c in clusters:
        new_groups.extend(gbs.split(c, ocg) if len(c) >= 2 else [c])

    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    colors = plt.cm.tab10.colors
    axes[0].scatter(pooled[:, 0], pooled[:, 1], s=3, color="gray", alpha=0.3, zorder=1)
    for i, piece in enumerate(old_pieces):
        plot_polygon(axes[0], piece, colors[i % len(colors)])
    axes[0].set_aspect("equal")
    axes[0].set_xticks([])
    axes[0].set_yticks([])
    axes[0].set_title(f"OLD: ChiOutline + BottleneckSplit\n-> {len(old_pieces)} piece(s) (true bunches: {n_bunches})")

    axes[1].set_aspect("equal")
    axes[1].set_xticks([])
    axes[1].set_yticks([])
    if render_alpha_outline:
        for i, group in enumerate(new_groups):
            color = colors[i % len(colors)]
            group_pts = np.vstack([object_points[j] for j in group])
            alpha_outline = AlphaOutline(group_pts, chi=2.0)
            for p in alpha_outline.compute():
                plot_polygon(axes[1], p["exterior"], color)
                for h in p["holes"]:
                    plot_polygon(axes[1], h, color, linewidth=1)
    else:
        plot_groups(axes[1], object_points, new_groups, "")
    axes[1].set_title(f"NEW: ObjectClusterGraph + GraphBottleneckSplit\n-> {len(new_groups)} group(s) "
                       f"(true bunches: {n_bunches})")

    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=4, continuous_update=False, description='n_bunches', max=6, min=2), IntS…

## Notes

- All methods here operate purely on 2D polygon/point geometry -- see each class's own docstring (`help(ChiOutline)`, etc.) for algorithm references and calibration caveats.
- The "old" pipeline's over-segmentation gets dramatically worse as `berries_per_bunch` grows or `berry_spread` shrinks (denser, more organic-looking clumps give the boundary-concavity-based criterion more false "necks" to find) -- try pushing those sliders to see it. The "new" pipeline's behavior is largely unaffected by either, since it never looks at berry *shape*, only berry *positions*.
- `ring_mode` only affects the full-pipeline comparison's first bunch -- compare the OLD panel (which cannot represent the hole) against the NEW panel with `render_alpha_outline` enabled.